# 🎧 Análisis de datos de Spotify

Integración y limpieza de tres bases de datos de Spotify para analizar
popularidad, atributos de audio y comportamiento de artistas.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


## 📌 Carga de datos

In [2]:
data_dir = Path("data")
artists_path = data_dir / "artists.csv"
songs_path = data_dir / "top-10k-spotify-songs-2025-07.csv"
songs_detail_path = data_dir / "top-10k-spotify-songs-2025-07-detailed.csv"

artists = pd.read_csv(artists_path)
songs = pd.read_csv(songs_path)
songs_detail = pd.read_csv(songs_detail_path)

artists.head()


,artist_id,name,followers,popularity,genres,main_genre
0,6Cqtx9fpxzggIMuKn0RGCp,Dream Supplier,13878,85,"sleep, white noise, rain, binaural, lullaby, c...","Easy Listening, Electronic, New Age, Rock"
1,2YZyLoL8N0Wb9xBt1NhZWg,Kendrick Lamar,45674032,91,"hip hop, west coast hip hop",Hip Hop
2,250b0Wlc5Vk0CoUsaCY84M,JENNIE,13013134,83,k-pop,Pop
3,6zO1dZ40fTZ5hY9NnnRJSk,yung kai,2117767,71,"k-pop, nyc pop, sped up, chill r&b, gen z sing...","Electronic, Pop, R&B, Rock"
4,4V8LLVI7PbaPR0K2TGSxFF,"Tyler, The Creator",25263506,88,"hip hop, rap",Hip Hop


## 🧾 Revisión rápida de tamaños

In [3]:
print('artists:', artists.shape)
print('songs:', songs.shape)
print('songs_detail:', songs_detail.shape)


artists: (5015, 6)
songs: (10000, 14)
songs_detail: (10000, 32)


## 🧹 Limpieza básica

In [4]:
artists.columns = [c.strip().lower() for c in artists.columns]
songs.columns = [c.strip().lower() for c in songs.columns]
songs_detail.columns = [c.strip().lower() for c in songs_detail.columns]

artists = artists.drop_duplicates()
songs = songs.drop_duplicates(subset=['track_id'])
songs_detail = songs_detail.drop_duplicates(subset=['track_id'])

artists['artist_id'] = artists['artist_id'].astype(str).str.strip()
songs_detail['track_id'] = songs_detail['track_id'].astype(str).str.strip()
songs_detail['artist_ids'] = songs_detail['artist_ids'].astype(str).str.strip()
songs['track_id'] = songs['track_id'].astype(str).str.strip()
songs['artist_ids'] = songs['artist_ids'].astype(str).str.strip()


## 🔗 Integración de datasets

In [5]:
artists_map = artists.rename(columns={
    'name': 'artist_name',
    'popularity': 'popularity_artist'
})

detail_exploded = songs_detail.copy()
detail_exploded['artist_ids_list'] = detail_exploded['artist_ids'].str.split(',')
detail_exploded = detail_exploded.explode('artist_ids_list')
detail_exploded['artist_ids_list'] = detail_exploded['artist_ids_list'].str.strip()

detail_with_artists = detail_exploded.merge(
    artists_map,
    left_on='artist_ids_list',
    right_on='artist_id',
    how='left'
)

detail_with_artists['followers'] = pd.to_numeric(detail_with_artists['followers'], errors='coerce')
detail_with_artists['followers_rank'] = detail_with_artists['followers'].fillna(-1)
idx = detail_with_artists.groupby('track_id')['followers_rank'].idxmax()
artist_best = detail_with_artists.loc[idx, [
    'track_id', 'artist_id', 'artist_name', 'followers', 'popularity_artist', 'genres', 'main_genre'
]]

songs_enriched = songs_detail.merge(
    artist_best,
    on='track_id',
    how='left'
)

songs_enriched = songs_enriched.merge(
    songs[['track_id', 'album_name', 'release_date', 'explicit', 'copies']],
    on='track_id',
    how='left',
    suffixes=('', '_basic')
)

songs_enriched.shape


(10000, 42)

## 🧽 Limpieza posterior a la integración

In [6]:
for col in ['album_name', 'release_date']:
    if col in songs_enriched.columns:
        songs_enriched[col] = songs_enriched[col].fillna('Unknown')

numeric_cols = [
    'popularity', 'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo',
    'duration_ms', 'total_artist_followers', 'avg_artist_popularity'
]

for col in numeric_cols:
    if col in songs_enriched.columns:
        songs_enriched[col] = pd.to_numeric(songs_enriched[col], errors='coerce')

for col in numeric_cols:
    if col in songs_enriched.columns:
        songs_enriched[col] = songs_enriched[col].fillna(songs_enriched[col].median())

songs_enriched.isna().sum().sort_values(ascending=False).head(10)


main_genre           3094
artist_name          3056
followers            3056
popularity_artist    3056
genres               3056
artist_id            3056
lyrics                417
key                    31
mode                   31
time_signature         31
dtype: int64

## ✅ Dataset integrado final

In [7]:
songs_enriched.head()


,rank,track_name,track_id,artist_names,artist_ids,album_name,album_id,popularity,duration,explicit,...,artist_id,artist_name,followers,popularity_artist,genres,main_genre,album_name_basic,release_date_basic,explicit_basic,copies_basic
0,1,Die With A Smile,2plbrEY59IikOBgBGLjaoe,Lady Gaga|Bruno Mars,1HY2Jd0NmPuamShAr6KMms|0du5cEVh5yTK9QJze8zA0C,Die With A Smile,10FLjwfpbxLmW8c25Xyc2N,100,4:11,0,...,NaN,NaN,NaN,NaN,NaN,NaN,Die With A Smile,2024-08-16,0,9
1,2,DtMF,3sK8wGT43QFpWrvNQsrQya,Bad Bunny,4q3ewBCX7sLwd24euuV69X,DeBÍ TiRAR MáS FOToS,5K79FLRUCSysQnVESLcTdb,98,3:57,1,...,4q3ewBCX7sLwd24euuV69X,Bad Bunny,105635803.0,99.0,"reggaeton, trap latino, urbano latino, latin",Latin,DeBÍ TiRAR MáS FOToS,2025-01-05,1,3
2,3,BIRDS OF A FEATHER,6dOtVTDdiauQNBQEDOtlAB,Billie Eilish,6qqNVTkY8uBg9cP3Jd7DAH,HIT ME HARD AND SOFT,7aJuG4TFXa2hmE4z1yxc3n,98,3:30,0,...,6qqNVTkY8uBg9cP3Jd7DAH,Billie Eilish,121278701.0,93.0,"art pop, pop",Pop,HIT ME HARD AND SOFT,2024-05-17,0,7
3,4,Clean Baby Sleep White Noise (Loopable),0zirWZTcXBBwGsevrsIpvT,Dream Supplier|Baby Sleeps|Background White Noise,6Cqtx9fpxzggIMuKn0RGCp|48zEowASEXWsK6lgP1xotZ|...,Best White Noise For Sleeping Baby,0NGHR9zjS5eFFlqtClA9VV,97,2:22,0,...,NaN,NaN,NaN,NaN,NaN,NaN,Best White Noise For Sleeping Baby,2020-04-29,0,2
4,5,BAILE INoLVIDABLE,2lTm559tuIvatlT1u0JYG2,Bad Bunny,4q3ewBCX7sLwd24euuV69X,DeBÍ TiRAR MáS FOToS,5K79FLRUCSysQnVESLcTdb,96,6:07,1,...,4q3ewBCX7sLwd24euuV69X,Bad Bunny,105635803.0,99.0,"reggaeton, trap latino, urbano latino, latin",Latin,DeBÍ TiRAR MáS FOToS,2025-01-05,1,1


## 📌 Pregunta 1: Artistas con mas followers

In [8]:
name_col = 'artist_name' if 'artist_name' in songs_enriched.columns else 'name'

top_artists = (
    songs_enriched[['artist_id', name_col, 'followers']]
    .dropna(subset=['artist_id'])
    .drop_duplicates(subset=['artist_id'])
    .sort_values('followers', ascending=False)
    .head(10)
)

top_artists


,artist_id,artist_name,followers
3087,4YRxDV8wJFPHPTeXepOstw,Arijit Singh,169306882.0
93,06HL4z0CvFAxyc27GXpf02,Taylor Swift,148838440.0
81,6eUKZXaKkcviH0Ku9w2n3V,Ed Sheeran,124012436.0
2,6qqNVTkY8uBg9cP3Jd7DAH,Billie Eilish,121278701.0
57,1Xyo4u8uXC1ZmMpatF05PJ,The Weeknd,116219359.0
133,66CXWjxzNUsdJxJ2JdwvnR,Ariana Grande,108715805.0
115,7dGJo4pcD2V6oG8kP0tJRR,Eminem,106214551.0
1,4q3ewBCX7sLwd24euuV69X,Bad Bunny,105635803.0
37,3TVXtAsR1Inumwj472S9r4,Drake,105373671.0
338,1uNFoZAHBGtllmzznpCI3s,Justin Bieber,86071004.0


## 📌 Pregunta 2: Generos mas populares

In [9]:
genre_col = None
if 'main_genres' in songs_enriched.columns:
    genre_col = 'main_genres'
elif 'main_genre' in songs_enriched.columns:
    genre_col = 'main_genre'

if genre_col is None:
    raise ValueError('No genre column found')

genre_series = songs_enriched[genre_col].dropna().astype(str)
genre_series = genre_series.str.split(',')
genre_exploded = genre_series.explode().str.strip()

top_genres_by_count = (
    genre_exploded.value_counts().head(10).rename_axis('genre').reset_index(name='tracks')
)

top_genres_by_popularity = (
    songs_enriched.assign(genre=songs_enriched[genre_col])
    .dropna(subset=['genre'])
    .assign(genre=lambda df: df['genre'].str.split(','))
    .explode('genre')
    .assign(genre=lambda df: df['genre'].str.strip())
    .groupby('genre', as_index=False)['popularity'].mean()
    .sort_values('popularity', ascending=False)
    .head(10)
)

top_genres_by_count, top_genres_by_popularity


(               genre  tracks
 0                Pop    4475
 1               Rock    2456
 2            Hip Hop    2294
 3              Latin    1916
 4                R&B    1321
 5  Traditional Music    1241
 6         Electronic     817
 7            Country     485
 8               Folk     223
 9              Metal     216,
          genre  popularity
 13         R&B   75.553369
 12         Pop   75.288715
 15        Rock   75.207248
 7      Hip Hop   74.978204
 5   Electronic   74.826193
 6         Folk   74.753363
 10       Metal   74.722222
 3      Country   74.684536
 9        Latin   74.538100
 2    Classical   74.259259)

## 📌 Pregunta 3: Albums lanzados a partir de 06/2025

In [10]:
release_col = 'release_date_basic' if 'release_date_basic' in songs_enriched.columns else 'release_date'

albums = (
    songs_enriched[['album_name', release_col]]
    .dropna(subset=['album_name'])
    .drop_duplicates()
)

albums[release_col] = pd.to_datetime(albums[release_col], errors='coerce')
start_date = pd.Timestamp('2025-06-01')
end_date = pd.Timestamp('2025-12-31')

filtered_albums = albums[(albums[release_col] >= start_date) & (albums[release_col] <= end_date)]

filtered_albums_sorted = filtered_albums.sort_values(release_col)
filtered_albums_sorted


,album_name,release_date_basic
1983,Mentiras: La Serie (Music from the Original TV...,2025-06-04
2458,Thug Life (Tamil),2025-06-04
4467,Apagar (Fundo Raso),2025-06-05
9193,"Sistemão, Vol. 1 (Ao Vivo)",2025-06-05
4466,Desapaixona Eu (Ao Vivo),2025-06-05
...,...,...
9177,Só Vivendo,2025-07-10
7736,Fica com Deus (Ao Vivo),2025-07-17
7737,KHE CALOR,2025-07-18
6472,Caos De Alguém (Ao Vivo),2025-07-18


### ✅ Conteo de albums por mes

In [11]:
albums_per_month = (
    filtered_albums_sorted
    .assign(month=filtered_albums_sorted[release_col].dt.to_period('M'))
    .groupby('month')
    .size()
    .reset_index(name='albums')
)

albums_per_month


,month,albums
0,2025-06,73
1,2025-07,8
2,2025-08,1


## 📌 Pregunta 4: Singles mas escuchados

In [12]:
singles = songs_enriched[songs_enriched['album_type'].str.lower() == 'single']

singles_top = (
    singles[['track_name', 'artist_names', 'popularity', 'rank']]
    .sort_values(['popularity', 'rank'], ascending=[False, True])
    .head(15)
)

singles_top


,track_name,artist_names,popularity,rank
0,Die With A Smile,Lady Gaga|Bruno Mars,100,1
3,Clean Baby Sleep White Noise (Loopable),Dream Supplier|Baby Sleeps|Background White Noise,97,4
5,Not Like Us,Kendrick Lamar,96,6
7,APT.,ROSÉ|Bruno Mars,95,8
9,Sailor Song,Gigi Perez,95,10
14,Ordinary,Alex Warren,94,15
15,"Good Luck, Babe!",Chappell Roan,94,16
16,Abracadabra,Lady Gaga,93,17
22,"One Of The Girls (with JENNIE, Lily Rose Depp)",The Weeknd|JENNIE|Lily-Rose Depp,92,23
23,Te Quería Ver,Alemán|Neton Vega,92,24


## 📌 Pregunta 5: Albums mas escuchados

In [13]:
albums_only = songs_enriched[songs_enriched['album_type'].str.lower() == 'album']

albums_top = (
    albums_only
    .dropna(subset=['album_name'])
    .groupby('album_name', as_index=False)
    .agg({
        'popularity': 'max',
        'rank': 'min',
        'artist_names': 'first'
    })
    .sort_values(['popularity', 'rank'], ascending=[False, True])
    .head(15)
)

albums_top


,album_name,popularity,rank,artist_names
866,DeBÍ TiRAR MáS FOToS,98,2,Bad Bunny
1411,HIT ME HARD AND SOFT,98,3,Billie Eilish
3378,The Secret of Us (Deluxe),96,7,Gracie Abrams
493,Black Panther The Album Music From And Inspire...,95,9,Kendrick Lamar|SZA
1310,GNX,94,12,Kendrick Lamar|SZA
1538,Hurry Up Tomorrow,93,21,The Weeknd|Playboi Carti
2789,Ruby,93,22,JENNIE
178,AM,92,26,Arctic Monkeys
263,Alligator Bites Never Heal,92,28,Doechii
2031,MUSIC,92,30,Playboi Carti
